# CaPy v2 — Phase 1 Training

Tri-modal contrastive learning for drug discovery.

**Phase 1 Gate:** bi-modal mol↔morph R@10 > 15%

**Runtime:** GPU (H100 recommended, T4/V100 also work)

In [ ]:
# Clone repo and install (idempotent — safe to re-run)
import os
if not os.path.exists("/content/CaPy-v2"):
    !git clone https://github.com/ogngnaoh/CaPy-v2.git
else:
    %cd /content/CaPy-v2
    !git pull
    %cd /content

%cd /content/CaPy-v2
!pip install -e ".[dev]" -q

# Verify install
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Download data (morphology, expression, metadata)
!python3 scripts/download.py

In [ ]:
# Preprocess: QC, normalize, scaffold split
!python3 scripts/preprocess.py

In [ ]:
# Quick sanity check — verify processed data
import json
from pathlib import Path

import pandas as pd

processed = Path("data/processed")
for split in ["train", "val", "test"]:
    df = pd.read_parquet(processed / f"{split}.parquet")
    print(f"{split}: {len(df)} treatments, {df.shape[1]} columns")

with open(processed / "feature_columns.json") as f:
    cols = json.load(f)
print(f"\nMorph features: {len(cols['morph_features'])}")
print(f"Expr features: {len(cols['expr_features'])}")

In [ ]:
# Optional: login to W&B for experiment tracking
# Skip this cell if you don't want W&B logging
import wandb
wandb.login()

In [ ]:
# Phase 1 gate: bi-modal mol-morph
# Expected: R@10 > 15% (random baseline ~10%)
!python3 scripts/train.py model=bi_mol_morph seed=42

In [ ]:
# After passing Phase 1 gate, train tri-modal
!python3 scripts/train.py model=tri_modal seed=42

In [ ]:
# Compare results from both runs
import torch

for name in ["bi_mol_morph_seed42", "tri_modal_seed42"]:
    path = f"checkpoints/{name}.pt"
    try:
        ckpt = torch.load(path, weights_only=False)
        print(f"{name}: best_metric={ckpt['best_metric']:.4f}, epoch={ckpt['epoch']}")
    except FileNotFoundError:
        print(f"{name}: not found (not trained yet)")

In [ ]:
# Multi-seed runs for the winning config
for seed in [42, 123, 456, 789, 1024]:
    print(f"\n{'='*60}")
    print(f"Training seed={seed}")
    print(f"{'='*60}")
    !python3 scripts/train.py model=bi_mol_morph seed={seed}